In [1]:
import sys
sys.path.append('../src')
from preprocessing import full_pipeline

X_train, y_train, stats = full_pipeline('../data/train.csv')
X_test, _, _ = full_pipeline('../data/test.csv', stats=stats)

print(X_train.shape, y_train.shape)
print(X_test.shape)

(1460, 94) (1460,)
(1459, 94)


In [2]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
import numpy as np

from preprocessing import build_preprocessor, build_preprocessor_scaled


In [3]:
preprocessor = build_preprocessor(X_train)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

In [4]:
scores = cross_val_score(model_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
rmse_scores = -scores
print("RMSE per fold:", rmse_scores)
print("Mean RMSE:", rmse_scores.mean())

RMSE per fold: [0.13387191 0.16537198 0.16594884 0.11770477 0.19082157]
Mean RMSE: 0.1547438141437212


## KNN

In [5]:
from sklearn.neighbors import KNeighborsRegressor

knn_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', KNeighborsRegressor())
])

knn_scores = cross_val_score(knn_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
knn_rmse = -knn_scores
print("RMSE per fold:", knn_rmse)
print("Mean RMSE:", knn_rmse.mean())

RMSE per fold: [0.17038534 0.18485514 0.17543754 0.16011428 0.17981534]
Mean RMSE: 0.17412152954231896


## KNN

The `preprocessor` above only scales the numeric columns (including the ordinal-encoded ones,
like ExterQual_encoded), leaving nominal columns unscaled:
- Nominal columns: one-hot encoded, values are 0 or 1
- Numeric/ordinal columns: scaled via StandardScaler, values are approximately -3 to 3

In [6]:
preprocessor_scaled = build_preprocessor_scaled(X_train)

knn_pipeline_scaled = Pipeline(steps=[
    ('preprocessor', preprocessor_scaled),
    ('model', KNeighborsRegressor())
])

knn_scores_scaled = cross_val_score(knn_pipeline_scaled, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
knn_rmse_scaled = -knn_scores_scaled
print("RMSE per fold:", knn_rmse_scaled)
print("Mean RMSE:", knn_rmse_scaled.mean())

RMSE per fold: [0.20316563 0.20444215 0.1913099  0.17643792 0.19692406]
Mean RMSE: 0.19445593061296867


## Random Forest

In [9]:
from sklearn.ensemble import RandomForestRegressor

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestRegressor(random_state=42))
])

rf_scores = cross_val_score(rf_pipeline, X_train, y_train, cv=5, scoring='neg_root_mean_squared_error')
rf_rmse = -rf_scores
print("RMSE per fold:", rf_rmse)
print("Mean RMSE:", rf_rmse.mean())

RMSE per fold: [0.1334012  0.14813141 0.14453949 0.13346181 0.15208001]
Mean RMSE: 0.1423227846109297


In [10]:
from sklearn.ensemble import GradientBoostingRegressor

gb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingRegressor())
])

gb_scores = cross_val_score(gb_pipeline, X_train, y_train, cv=5,scoring='neg_root_mean_squared_error')
gb_rmse = -gb_scores
print("RMSE per fold:", gb_rmse)
print("Mean RMSE:", gb_rmse.mean())

RMSE per fold: [0.11320129 0.13620256 0.12824727 0.11759728 0.12280137]
Mean RMSE: 0.12360995708577249


## Phase 5: Baseline Model Comparison

Built one Pipeline per model (preprocessor + model), evaluated all four with 
identical 5-fold cross-validation on log1p(SalePrice), scoring on RMSE.

| Model              | Mean RMSE | Fold spread     |
|---------------------|-----------|-----------------|
| Linear Regression    | 0.1547    | 0.118 - 0.191   |
| KNN                  | 0.1741    | 0.160 - 0.185   |
| Random Forest         | 0.1423    | 0.133 - 0.152   |
| Gradient Boosting     | 0.1236    | 0.113 - 0.136   |

**Winner: Gradient Boosting** - best mean RMSE and tightest fold spread
(most consistent across different data splits).

### Side experiment: does scaling one-hot columns help KNN?
Built a second preprocessor (build_preprocessor_scaled) that also scales the
one-hot encoded nominal columns (using StandardScaler(with_mean=False)), 
hypothesizing that the scale mismatch between 0/1 one-hot columns and 
-3 to 3 scaled numeric columns was hurting KNN's distance calculations.

Result: WORSE, not better (0.1945 vs 0.1741 baseline). 
Likely cause: with_mean=False divides by standard deviation without centering - 
for rare categories (e.g. Neighborhood values with only 2-3 houses), this 
produces very large scaled values instead of a tidy -3 to 3 range, actively 
distorting distance calculations for sparse categories rather than fixing them.
Conclusion: keep the original mixed-scaling preprocessor for all models going forward.

### Next steps
- Test is_outlier flag: compare performance with vs without the 2 flagged 
  outlier rows (523, 1298) from Phase 1/2
- Try xgboost/lightgbm as stronger gradient boosting alternatives
- Hyperparameter tuning on the best candidate(s)